# TATR / TMTR — Reduced Replication
## FTS-Diffusion Paper, Section 5.3 / Fig. 6

### Multi-asset, multi-experiment notebook
Supports: **S&P 500**, **GOOG**, **ZC=F** for both **TATR** and **TMTR** experiments.

### Configuration (reduced from paper to fit Colab Pro A100)
- **15 runs** (paper: 100) — for 95% CI estimate
- **101 augmentation steps × 252 days = 100 years** of synthetic data per run (matches paper)
- **Train LSTM at 11 evaluation points**: 0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100 years
- **Total LSTMs trained**: 15 × 11 = **165** per asset (paper: ~10,100)

### Persistent storage on Drive
```
/content/drive/MyDrive/fts_diffusion/
├── architectures/{asset}/   # Pre-trained FTS-Diffusion per asset
├── synthetic/{asset}/       # Generated synthetic series per run
├── results/{exp}/{asset}/   # MAPE CSVs + summary
└── figures/{exp}/           # Final plots
```

### Paper-faithful hyperparameters
| Param | Value |
|-------|-------|
| LSTM hidden_dim | 32 |
| LSTM layers | 2 |
| Window size | 64 |
| Loss | MAE |
| Optimizer | Adam, lr=1e-2 |
| Epochs | 100 |
| Training mode | **Full-batch** |

## 1. Environment Setup

In [ ]:
import os, sys, subprocess, time, json, shutil
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/thaidinger/ml-in-fcs.git'
    BRANCH = 'deli_work'
    CLONE_DIR = '/content/ml-in-fcs'
    if not os.path.exists(CLONE_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, CLONE_DIR])
    else:
        subprocess.check_call(['git', '-C', CLONE_DIR, 'pull'])

    REPO_ROOT = f'{CLONE_DIR}/fts-diffusion-ref'
    PERSIST_ROOT = '/content/drive/MyDrive/fts_diffusion'
else:
    REPO_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref'
    PERSIST_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts_diffusion_run'

# Create top-level Drive structure (subdirs created lazily once ASSET/EXPERIMENT chosen)
os.makedirs(PERSIST_ROOT, exist_ok=True)
for sub in ['architectures', 'synthetic', 'results', 'figures']:
    os.makedirs(f'{PERSIST_ROOT}/{sub}', exist_ok=True)

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
for d in ['data', 'res', 'trained_models', 'figs']:
    os.makedirs(d, exist_ok=True)

print(f"Working dir:    {os.getcwd()}")
print(f"Persistent dir: {PERSIST_ROOT}")

In [ ]:
# Install dependencies
for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy', 'scikit-learn', 'torch', 'tqdm', 'tslearn', 'dtaidistance', 'yfinance']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("\u2713 Deps ready")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU info
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # For maximum reproducibility (slightly slower):
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("\u2713 Deterministic mode ON (TF32 disabled)")

# Authors' modules
from utils.load_data import get_fts, load_actual_fts
from models.model_params import prm_params, pgm_params, pem_params
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)
from models.sampling import generate_timeseries_ftsdiffusion
from experiments.utils_downstream import (
    setup_dowmstream_tatr, init_first_segment,
    Timeseries2Dataset_Downstream, create_syn_dataset,
    concat_datasets_downstream, copy_dataset_downstream)
from experiments.predictor_lstm import LSTMPredictor, set_loss_fn

print("\u2713 Imports ready")

## 2. Configuration

In [ ]:
# ============================================================================
# === EXPERIMENT CONFIG ===
# Change ASSET, K, EXPERIMENT, PROTOCOL to switch experiment dimension.
# All Drive paths are derived automatically.
# ============================================================================

ASSET          = 'sp500'      # 'sp500' | 'goog' | 'zcf'
K              = 14           # number of SISC clusters (paper uses 14; try 11, 12, etc.)
EXPERIMENT     = 'tatr'       # 'tatr' | 'tmtr'
PROTOCOL       = 'authors'    # 'authors' | 'single' | 'random_init' | 'split' | 'burn_in' (B-year burn-in + Y-year chunks)
B_YEARS        = 30           # burn-in years (only used by 'burn_in' protocol)
Y_YEARS        = 3            # chunk length in years (only used by 'burn_in' protocol)

# Per-asset metadata (matches paper Appendix E.1)
ASSETS_CONFIG = {
    'sp500':    {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500'},
    'sp500_us': {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500 (retrained from scratch)'},
    'goog':  {'ticker': 'GOOG',  'start': '2005-01-01', 'end': '2020-01-01', 'pretty': 'GOOG'},
    'zcf':   {'ticker': 'ZC=F',  'start': '2001-01-01', 'end': '2020-01-01', 'pretty': 'ZC=F (Corn futures)'},
}
assert ASSET in ASSETS_CONFIG, f"Unknown asset {ASSET}"
assert EXPERIMENT in ['tatr', 'tmtr'], f"Unknown experiment {EXPERIMENT}"
assert PROTOCOL in ['authors', 'single', 'random_init', 'split', 'burn_in'], f"Unknown protocol {PROTOCOL}"
assert isinstance(K, int) and K >= 2, f"K must be a positive integer >= 2, got {K}"

DATANAME       = ASSET
TICKER         = ASSETS_CONFIG[ASSET]['ticker']
START_DATE     = ASSETS_CONFIG[ASSET]['start']
END_DATE       = ASSETS_CONFIG[ASSET]['end']
PRETTY_NAME    = ASSETS_CONFIG[ASSET]['pretty']

# === Persistent paths derived from ASSET / K / EXPERIMENT / PROTOCOL ===
# Structure on Drive:
#   fts_diffusion/architectures/{ASSET}/k{K}/{res,trained_models,data}
#   fts_diffusion/synthetic/{ASSET}/k{K}/{PROTOCOL}/run_XX_*.npy
#   fts_diffusion/results/{EXPERIMENT}/{ASSET}/k{K}/{PROTOCOL}/run_XX.csv
#   fts_diffusion/figures/{EXPERIMENT}/{ASSET}/k{K}/{PROTOCOL}/{live,final}.{png,pdf}
ARCH_DIR    = f'{PERSIST_ROOT}/architectures/{ASSET}/k{K}'
SYN_DIR     = f'{PERSIST_ROOT}/synthetic/{ASSET}/k{K}/{PROTOCOL}'
RES_DIR     = f'{PERSIST_ROOT}/results/{EXPERIMENT}/{ASSET}/k{K}/{PROTOCOL}'
FIG_DIR     = f'{PERSIST_ROOT}/figures/{EXPERIMENT}/{ASSET}/k{K}/{PROTOCOL}'
for d in [ARCH_DIR, SYN_DIR, RES_DIR, FIG_DIR,
          f'{ARCH_DIR}/res', f'{ARCH_DIR}/trained_models', f'{ARCH_DIR}/data']:
    os.makedirs(d, exist_ok=True)

# === Override the global prm_params ===
from models.model_params import prm_params, pgm_params, pem_params
prm_params['dataname'] = ASSET
prm_params['k'] = K
# These derived params reference prm_params['k'] at module load time, so we must update them too:
pgm_params['n_patterns'] = K
pem_params['n_patterns'] = K

# === Experiment hyperparameters ===
N_RUNS         = 30

# EVAL_YEARS: which augmentation lengths to evaluate the LSTM at.
# Mutable: add/remove years between sessions, already-saved years are loaded from disk.
EVAL_YEARS     = [0, 5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100]

# SYN_MAX_YEARS: how much synthetic data to generate ONCE per run.
# As long as max(EVAL_YEARS) <= SYN_MAX_YEARS, no synthetic regeneration is needed.
SYN_MAX_YEARS  = 100
AUG_LENGTH     = 252

assert max(EVAL_YEARS) <= SYN_MAX_YEARS

# === LSTM CONFIG (paper-faithful) ===
WINDOW_SIZE    = 64
LSTM_HIDDEN    = 32
LSTM_LAYERS    = 2
AHEAD          = 1
N_EPOCHS       = 100
LR             = 1e-2
LSTM_LOSS      = 'mae'
DATATYPE       = 'prices'

print(f"Configuration:")
print(f"  ASSET         = {ASSET}  ({PRETTY_NAME})")
print(f"  K (clusters)  = {K}")
print(f"  EXPERIMENT    = {EXPERIMENT.upper()}")
print(f"  PROTOCOL      = {PROTOCOL!r}")
if PROTOCOL == 'burn_in':
    print(f"    burn_in: B={B_YEARS} years, Y={Y_YEARS} years")
print(f"  N_RUNS        = {N_RUNS}")
print(f"  EVAL_YEARS    = {EVAL_YEARS}")
print(f"  Total LSTMs   = {N_RUNS * len(EVAL_YEARS)} (this session)")
print()
print(f"Drive paths (now K-aware):")
print(f"  ARCH_DIR = {ARCH_DIR}")
print(f"  SYN_DIR  = {SYN_DIR}")
print(f"  RES_DIR  = {RES_DIR}")
print(f"  FIG_DIR  = {FIG_DIR}")

## 3. Phase 1 — Train FTS-Diffusion Architecture (Once)

This step is **expensive (~1 hour on A100)** but only runs once. Checkpoints persist on Drive.

In [ ]:
# Sync persistent checkpoints to working dirs (so train_ftsdiffusion finds them)

def _is_for_asset(filename, asset, kind):
    """Return True if filename belongs to the given asset."""
    if kind == 'res':
        # SISC artifacts: sisc_{asset}_k{K}_l{lmin}-{lmax}_*.csv
        return filename.startswith(f'sisc_{asset}_') and filename.endswith('.csv')
    if kind == 'trained_models':
        # PGM/PEM filenames contain the asset name (e.g. pgm-2_c48-80_sp500_*)
        return (f'_{asset}_' in filename) and (filename.endswith('.pth') or filename.endswith('.pt'))
    if kind == 'data':
        return filename == f'{asset}_timeseries.csv'
    return False


def sync_persistent_to_working(persistent_dir, working_subdir, asset=None, kind=None):
    """Copy from Drive into working dir if working dir is empty (per file).
    Optionally filter by asset/kind to avoid cross-asset contamination."""
    src = persistent_dir
    dst = os.path.join(REPO_ROOT, working_subdir)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        if not os.path.exists(dst_f):
            shutil.copy(src_f, dst_f)
            print(f"  Restored {f}")


def sync_working_to_persistent(working_subdir, persistent_dir, asset=None, kind=None):
    """Copy from working dir to Drive (overwrite). Filter by asset/kind for clean separation."""
    src = os.path.join(REPO_ROOT, working_subdir)
    dst = persistent_dir
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        shutil.copy(src_f, dst_f)


# Restore architecture checkpoints if they exist on Drive (filtered by current ASSET)
if ASSET == 'sp500_us':
    print(f"[Note] ASSET=sp500_us: we will train FTS-Diffusion from scratch")
    print(f"       (no pre-trained checkpoints will be pulled from fts-diffusion-ref/)")
print(f"Checking for existing {ASSET} architecture checkpoints on Drive...")
sync_persistent_to_working(f'{ARCH_DIR}/res',             'res',             asset=ASSET, kind='res')
sync_persistent_to_working(f'{ARCH_DIR}/trained_models',  'trained_models',  asset=ASSET, kind='trained_models')
sync_persistent_to_working(f'{ARCH_DIR}/data',            'data',            asset=ASSET, kind='data')

In [ ]:
# Step 1: Download asset data if not already present
import shutil

ts_path = f'data/{DATANAME}_timeseries.csv'

# Special handling for sp500_us: reuse sp500's CSV (same ticker + dates, no need to re-download)
if ASSET == 'sp500_us':
    src_path = 'data/sp500_timeseries.csv'
    if os.path.exists(src_path) and not os.path.exists(ts_path):
        shutil.copy(src_path, ts_path)
        print(f"\u2713 Reused {src_path} \u2192 {ts_path} (sp500_us shares data with sp500)")

if not os.path.exists(ts_path):
    print(f"Downloading {TICKER} from {START_DATE} to {END_DATE}...")
    get_fts(ticker=TICKER, fts_name=DATANAME, start_date=START_DATE, end_date=END_DATE)
else:
    print(f"\u2713 Data already at {ts_path}")

fts = load_actual_fts(DATANAME).squeeze()
print(f"{PRETTY_NAME} series: {len(fts)} points, range [{fts.min():.4f}, {fts.max():.4f}]")


### Auto-fix: PEM device bug (apply BEFORE Phase 1 training)

The authors' `pattern_evolution_module.train_evolution_module` does not move target tensors to GPU before computing `cross_entropy`, causing a CUDA device-mismatch error on most modern PyTorch versions. This cell patches the source file in the cloned repo before any architecture training.

**Idempotent**: safe to re-run; does nothing if already patched. Keep this cell in the notebook permanently — it will be applied automatically every Colab session (since the patch lives on the cloned repo, which is reset on disconnect).

In [ ]:
# === AUTO-FIX: PEM target tensors not moved to GPU (idempotent) ===
import sys, importlib

pem_file = f'{REPO_ROOT}/models/pattern_evolution_module.py'
PATCH_MARKER = '_FTS_PATCH_DEVICE_FIX_'

with open(pem_file) as f:
    code = f.read()

if PATCH_MARKER in code:
    print("\u2713 PEM file already patched")
else:
    old_block = (
        "      target_pattern = batch_y[:, 0].long()\n"
        "      target_length = (batch_y[:, 1] - 10).long()\n"
        "      target_magnitude = batch_y[:, 2].float().view(-1, 1)"
    )
    new_block = (
        "      # " + PATCH_MARKER + "\n"
        "      _dev = next(model.parameters()).device\n"
        "      batch_x = batch_x.to(_dev)\n"
        "      target_pattern = batch_y[:, 0].long().to(_dev)\n"
        "      target_length = (batch_y[:, 1] - 10).long().to(_dev)\n"
        "      target_magnitude = batch_y[:, 2].float().view(-1, 1).to(_dev)"
    )
    if old_block not in code:
        print("\u26a0 Pattern not found \u2014 file may have changed. Manual inspection required.")
    else:
        code = code.replace(old_block, new_block)
        with open(pem_file, 'w') as f:
            f.write(code)
        print(f"\u2713 Patched {pem_file}")

# Verify patch is in file
with open(pem_file) as f:
    content = f.read()
assert PATCH_MARKER in content, "PATCH NOT APPLIED"

# Aggressive cache purge to ensure reloaded modules pick up the new code
mods_to_purge = [m for m in list(sys.modules.keys())
                 if any(p in m for p in [
                     'pattern_evolution_module',
                     'pattern_recognition_module',
                     'pattern_generation_module',
                     'pattern_conditioned_diffusion',
                     'scaling_autoencoder',
                     'train_ftsdiffusion',
                     'load_models',
                 ])]
for m in mods_to_purge:
    del sys.modules[m]

# Re-import fresh
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)

# Verify the loaded function contains the patch
import models.pattern_evolution_module as pem_mod
import inspect
src_code = inspect.getsource(pem_mod.train_evolution_module)
assert PATCH_MARKER in src_code, "Loaded function does NOT have patch \u2014 try Runtime > Restart runtime"

print("\u2713 Modules reloaded; patched function is active")

In [ ]:
# Step 2: Train SISC + PGM + PEM (only if no checkpoint for this asset)
has_all = (_has_recognition_artifacts() and
           _has_generation_artifact() and
           _has_evolution_artifact())

if has_all:
    print(f"\u2713 All {ASSET} architecture artifacts found \u2014 skipping training.")
else:
    print(f"Training FTS-Diffusion architecture for {PRETTY_NAME} (SISC + PGM + PEM)...")
    print("Estimated time: ~50-75 min on A100")
    t0 = time.time()
    train_ftsdiffusion(fts, train_test_split=0.8, store_model=True)
    elapsed = (time.time() - t0) / 60
    print(f"\u2713 Architecture training complete in {elapsed:.1f} min")

# Persist asset-specific checkpoints to Drive
print(f"Syncing {ASSET} checkpoints to Drive...")
sync_working_to_persistent('res', f'{ARCH_DIR}/res', asset=ASSET, kind='res')
sync_working_to_persistent('trained_models', f'{ARCH_DIR}/trained_models', asset=ASSET, kind='trained_models')
sync_working_to_persistent('data', f'{ARCH_DIR}/data', asset=ASSET, kind='data')
print(f"\u2713 {ASSET} checkpoints persisted on Drive at {ARCH_DIR}.")

## 3.5 Visualizza i pattern SISC appresi

Dopo Phase 1 (allenamento dell'architettura), SISC ha discovered K=14 pattern ricorrenti dalla serie storica. Visualizziamoli.

In [ ]:
# === SISC: visualizza i K pattern appresi ===
import pandas as pd
import matplotlib.pyplot as plt

# K already defined in config cell
sisc_prefix = f'res/sisc_{ASSET}_k{K}_l{prm_params["l_min"]}-{prm_params["l_max"]}_dba_kmpp'

centroids_path = f'{sisc_prefix}_centroids.csv'
labels_path = f'{sisc_prefix}_labels.csv'
segmentation_path = f'{sisc_prefix}_segmentation.csv'

if not all(os.path.exists(p) for p in [centroids_path, labels_path, segmentation_path]):
    print(f"\u26a0 SISC artifacts not found in {sisc_prefix}_*.csv")
    print(f"  Run Phase 1 first (or sync from Drive)")
else:
    centroids = pd.read_csv(centroids_path).values[:, 1:]   # (K, l_max)
    labels = pd.read_csv(labels_path).values[:, 1].astype(int)
    segmentation = pd.read_csv(segmentation_path).values[:, 1].astype(int)
    seg_lengths = np.diff(segmentation)

    # === 1) Plot: K pattern appresi ===
    n_cols = 5
    n_rows = (K + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2.5*n_rows))
    axes = axes.flatten()

    # Color palette: each pattern gets its own color
    cmap = plt.cm.get_cmap('tab20', K)

    for k in range(K):
        ax = axes[k]
        ax.plot(centroids[k], linewidth=2, color=cmap(k))
        ax.set_title(f'Pattern {k}  (n={int(np.sum(labels == k))} segs)', fontsize=10)
        ax.grid(alpha=0.3)
        ax.set_xticks([])

    # Hide unused subplots
    for j in range(K, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'SISC patterns learned for {PRETTY_NAME} (K={K})',
                 fontsize=13, fontweight='bold', y=1.005)
    plt.tight_layout()

    # Save
    sisc_fig_dir = f'{PERSIST_ROOT}/figures/sisc/{ASSET}/k{K}'
    os.makedirs(sisc_fig_dir, exist_ok=True)
    fig.savefig(f'{sisc_fig_dir}/patterns.pdf', bbox_inches='tight', dpi=150)
    fig.savefig(f'{sisc_fig_dir}/patterns.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\u2713 Saved SISC patterns to {sisc_fig_dir}/patterns.{{pdf,png}}")

    # === 2) Statistiche di frequenza dei pattern ===
    print(f"\nPattern frequency in {PRETTY_NAME} segmentation:")
    print(f"  Total segments: {len(labels)}")
    print(f"  Total covered days: {seg_lengths.sum()}")
    print(f"  Mean segment length: {seg_lengths.mean():.2f} days")
    print(f"  Pattern occurrences: {np.bincount(labels, minlength=K).tolist()}")

    # === 3) Plot: distribuzione di frequenza ===
    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))

    # Bar plot: frequenza dei pattern
    counts = np.bincount(labels, minlength=K)
    axes2[0].bar(range(K), counts, color=[cmap(k) for k in range(K)])
    axes2[0].set_xlabel('Pattern ID')
    axes2[0].set_ylabel('Number of segments')
    axes2[0].set_title(f'Pattern frequency ({PRETTY_NAME})')
    axes2[0].set_xticks(range(K))
    axes2[0].grid(alpha=0.3, axis='y')

    # Hist: distribuzione delle lunghezze dei segmenti
    axes2[1].hist(seg_lengths, bins=range(int(seg_lengths.min()), int(seg_lengths.max())+2),
                  edgecolor='black', alpha=0.7, color='steelblue')
    axes2[1].set_xlabel('Segment length (days)')
    axes2[1].set_ylabel('Frequency')
    axes2[1].set_title(f'Segment length distribution ({PRETTY_NAME})')
    axes2[1].grid(alpha=0.3, axis='y')

    plt.tight_layout()
    fig2.savefig(f'{sisc_fig_dir}/pattern_stats.pdf', bbox_inches='tight', dpi=150)
    fig2.savefig(f'{sisc_fig_dir}/pattern_stats.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig2)
    print(f"\u2713 Saved SISC stats to {sisc_fig_dir}/pattern_stats.{{pdf,png}}")

## 4. Phase 2 — TATR Loop

For each of the 15 runs:
1. Generate `MAX_YEARS × 252 = 25,200` days of synthetic data (~12 min on A100)
2. For each evaluation year in `[0, 10, 20, ..., 100]`:
   - Build augmented dataset = real init + synthetic [0:year × 252]
   - Train LSTM (full-batch, 100 epochs, lr=1e-2, MAE loss, hidden=32)
   - Test on real test set → record MAPE
3. Save run results to CSV after each run.

In [ ]:
# === Setup TATR datasets (ADAPTIVE: handles short datasets like GOOG/ZC=F) ===
#
# The authors' setup_dowmstream_tatr() hardcodes init_period = 252*5 days (5 years),
# which corresponds to ~62.5% of the test set length for S&P 500 (5y init / 8y test).
# For shorter datasets (GOOG, ZC=F) the hardcoded value exceeds the available test data
# and the code crashes.
#
# To stay paper-faithful, we keep the same init/test RATIO as S&P 500 in the paper:
#     init_fraction = 5 / 8 = 0.625
# Then the absolute init period scales with each asset's available test data.
# This keeps S&P 500 identical to the paper (1260 days / 756 days) and provides a
# principled, documented choice for GOOG and ZC=F.

from experiments.utils_downstream import (
    get_downstream_data, init_first_segment, init_tatr_set,
    Timeseries2Dataset_Downstream
)


# Per-asset init/test split. Default = 0.625 (= 5/8, same as paper S&P 500).
INIT_FRACTION_PER_ASSET = {
    'sp500': 0.625,
    'goog':  0.625,
    'zcf':   0.625,
}


def setup_dowmstream_tatr_adaptive(window_size, init_fraction=0.625, min_test_years=1):
    """Adaptive TATR setup that uses init_fraction of the test data as initial LSTM training set."""
    downstream_timeseries, segments_test, labels_test, lengths_test = get_downstream_data()
    total_len = len(downstream_timeseries)

    init_period = int(total_len * init_fraction)
    max_init = total_len - 252 * min_test_years
    init_period = min(init_period, max_init)
    init_period = max(init_period, 252)

    init_timeseries = downstream_timeseries[:init_period]
    test_timeseries = downstream_timeseries[init_period:]

    init_dataset, scaler = init_tatr_set(init_timeseries, window_size)
    first_segment = init_first_segment(segments_test, labels_test, lengths_test)
    test_dataset = init_tatr_set(test_timeseries, window_size, scaler)

    # === Split SEGMENTS_TEST at the same boundary as init_timeseries / test_timeseries ===
    # Segments are in temporal order; cumulate their lengths to find the boundary.
    cum = 0
    boundary_idx = 0
    for i, l in enumerate(lengths_test):
        if cum + l <= init_period:
            cum += l
            boundary_idx = i + 1
        else:
            break

    segments_init = segments_test[:boundary_idx]
    labels_init   = labels_test[:boundary_idx]
    lengths_init  = lengths_test[:boundary_idx]

    print(f"Adaptive TATR setup for {ASSET} ({PRETTY_NAME}):")
    print(f"  Total downstream timeseries: {total_len} days ({total_len/252:.2f} years)")
    print(f"  init_fraction:               {init_fraction:.3f}")
    print(f"  Init period (LSTM training): {init_period} days ({init_period/252:.2f} years)")
    print(f"  Test period (LSTM eval):     {total_len - init_period} days ({(total_len-init_period)/252:.2f} years)")
    print(f"  init_dataset shape:  {init_dataset.shape}")
    print(f"  test_dataset shape:  {test_dataset.shape}")
    print(f"  segments_init: {len(segments_init)} segments (covering {cum} days of LSTM-train period)")
    return (init_dataset, first_segment, test_dataset, scaler,
            segments_init, labels_init, lengths_init)


init_fraction = INIT_FRACTION_PER_ASSET.get(ASSET, 0.625)
(init_dataset, first_segment, test_dataset, scaler,
 SEGMENTS_INIT, LABELS_INIT, LENGTHS_INIT) = setup_dowmstream_tatr_adaptive(
    WINDOW_SIZE, init_fraction=init_fraction)
init_state, init_segment = first_segment

In [ ]:
# === LSTM training & evaluation functions (paper-faithful) ===

def train_lstm_full_batch(dataset, n_epochs=N_EPOCHS, hidden_dim=LSTM_HIDDEN,
                          n_layers=LSTM_LAYERS, output_dim=AHEAD, criterion_str=LSTM_LOSS,
                          lr=LR, seed=0):
    """Match the authors' full-batch training in predictor_lstm.py exactly."""
    torch.manual_seed(seed)
    model = LSTMPredictor(input_dim=1, hidden_dim=hidden_dim, output_dim=output_dim,
                          n_layers=n_layers, device=device).to(device)
    criterion = set_loss_fn(criterion_str)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    X = dataset[:, :-output_dim].unsqueeze(-1).to(device)
    y = dataset[:, -output_dim:].to(device)
    
    for epoch in range(n_epochs):
        model.train()
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return model, float(loss.item())


def evaluate_lstm(model, test_dataset, scaler, output_dim=AHEAD):
    """Compute MAPE on the real test set, on the original (un-scaled) scale."""
    model.eval()
    with torch.no_grad():
        X = test_dataset[:, :-output_dim].unsqueeze(-1).to(device)
        y_true = test_dataset[:, -output_dim:].numpy()
        y_pred = model(X).cpu().numpy()
        # Inverse-transform back to original scale
        y_true_orig = scaler.inverse_transform(y_true)
        y_pred_orig = scaler.inverse_transform(y_pred)
    return float(MAPE(y_true_orig, y_pred_orig))

print("\u2713 Train/eval functions ready")

In [ ]:
# === TATR run for a single seed (4 protocols, MC state tracking for 'single' & 'split') ===

@torch.no_grad()
def generate_with_states(T, init_state, init_segment):
    """Like generate_timeseries_ftsdiffusion but also returns the (p, l, m) state sequence.

    Returns:
        timeseries: 1D float32 array of length T (the synthetic price series)
        states_arr: 2D float32 array of shape (n_segments+1, 3) where each row is (pattern, length, magnitude).
                    The first row corresponds to the initial state (init_segment).
    """
    from models.load_models import load_ftsdiffusion
    from models.utils_sampling import sampling_inputs
    from models.sampling import state_evolution_ftsdiffusion, segment_generation_ftsdiffusion

    model = load_ftsdiffusion()
    l_min = prm_params['l_min']
    _, _, patterns = sampling_inputs()
    dev = next(model['evolution'].parameters()).device
    patterns = torch.tensor(patterns, dtype=torch.float32).to(dev)

    timeseries = list(init_segment)
    if hasattr(init_state, 'to'):
        state = init_state.to(dev)
    else:
        state = torch.tensor(init_state, dtype=torch.float32).to(dev).view(1, -1)

    states_list = [state.squeeze(0).cpu().numpy().astype(np.float32).copy()]
    curr_T = len(timeseries)

    while curr_T < T:
        state = state_evolution_ftsdiffusion(model['evolution'], state, l_min)
        segment = segment_generation_ftsdiffusion(model['generation'], state, patterns)
        segment = segment - segment[0] + timeseries[-1]
        seg_len = len(segment)
        timeseries.extend(segment)
        curr_T += seg_len
        states_list.append(state.squeeze(0).cpu().numpy().astype(np.float32).copy())

    timeseries_arr = np.array(timeseries[:T], dtype=np.float32)
    states_arr = np.stack(states_list, axis=0)
    return timeseries_arr, states_arr


def run_single_tatr(run_idx, eval_years, seed_base=42, protocol=None):
    """Run one TATR replicate with the chosen protocol.

    Protocols:
      'authors':     100 fresh 252-day blocks, each from same fixed (init_state, init_segment)
      'single':      1 continuous SYN_MAX_YEARS-year trajectory, prefixes for each year (TRACKS MC STATES)
      'random_init': 100 blocks, each from a random init_segment sampled from SEGMENTS_INIT
      'split':       1 continuous trajectory, split into 252-day blocks unfolded separately (TRACKS MC STATES)
    """
    if protocol is None:
        protocol = PROTOCOL

    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'

    results = {}
    if os.path.exists(run_csv):
        df_prev = pd.read_csv(run_csv)
        results = dict(zip(df_prev['eval_year'].astype(int), df_prev['mape'].astype(float)))
        if set(eval_years).issubset(set(results.keys())):
            print(f"  \u2713 Run {run_idx}: all requested years already in CSV \u2014 skipping.")
            return {ey: results[ey] for ey in eval_years}
        elif results:
            done = sorted(results.keys())
            missing = sorted(set(eval_years) - set(done))
            print(f"  \u26a0 Run {run_idx} partial: years already saved={done}, to compute={missing}")

    np.random.seed(seed_base + run_idx)
    torch.manual_seed(seed_base + run_idx)

    # ========================================================================
    # PROTOCOL: 'authors'
    # ========================================================================
    if protocol == 'authors':
        blocks_path = f'{SYN_DIR}/run_{run_idx:02d}_blocks.npy'
        n_blocks_needed = max(eval_years)

        cached_blocks = None
        if os.path.exists(blocks_path):
            cached_blocks = np.load(blocks_path)
            n_existing = cached_blocks.shape[0] if cached_blocks.ndim == 2 else 0
            print(f"  Loading {n_existing} cached synthetic blocks for run {run_idx} (protocol={protocol})")
        else:
            n_existing = 0

        if n_existing < n_blocks_needed:
            n_to_generate = n_blocks_needed - n_existing
            print(f"  Generating {n_to_generate} new 252-day blocks (~{n_to_generate*0.05:.1f} min)...")
            t0 = time.time()
            new_blocks = []
            for b in range(n_existing, n_blocks_needed):
                block = generate_timeseries_ftsdiffusion(
                    T=AUG_LENGTH,
                    init_state=init_state,
                    init_segment=init_segment)
                block = np.asarray(block, dtype=np.float32)[:AUG_LENGTH]
                if len(block) < AUG_LENGTH:
                    block = np.concatenate([block, np.full(AUG_LENGTH - len(block), block[-1] if len(block) > 0 else 0.0)])
                new_blocks.append(block)
                if (b + 1) % 10 == 0:
                    stacked = np.stack(new_blocks, axis=0)
                    full = np.concatenate([cached_blocks, stacked], axis=0) if cached_blocks is not None else stacked
                    np.save(blocks_path, full)
            new_blocks_arr = np.stack(new_blocks, axis=0)
            all_blocks = np.concatenate([cached_blocks, new_blocks_arr], axis=0) if cached_blocks is not None else new_blocks_arr
            np.save(blocks_path, all_blocks)
            print(f"  Block generation: {(time.time()-t0)/60:.1f} min")
        else:
            all_blocks = cached_blocks

        def build_curr_dataset(ey):
            if ey == 0:
                return copy_dataset_downstream(init_dataset)
            curr = copy_dataset_downstream(init_dataset)
            for b in range(ey):
                block = all_blocks[b]
                block_dataset = create_syn_dataset(block, WINDOW_SIZE, scaler, DATATYPE)
                curr = concat_datasets_downstream(curr, block_dataset)
            return curr

    # ========================================================================
    # PROTOCOL: 'single' \u2014 1 continuous trajectory + STATE TRACKING
    # ========================================================================
    elif protocol == 'single':
        syn_path = f'{SYN_DIR}/run_{run_idx:02d}_syn.npy'
        states_path = f'{SYN_DIR}/run_{run_idx:02d}_states.npy'
        needed_length = SYN_MAX_YEARS * AUG_LENGTH

        full_syn = None
        if os.path.exists(syn_path) and os.path.exists(states_path):
            existing = np.load(syn_path)
            if len(existing) >= needed_length:
                print(f"  Loading saved trajectory + states for run {run_idx} (protocol={protocol})")
                full_syn = existing

        if full_syn is None:
            print(f"  Generating one {SYN_MAX_YEARS}-year continuous trajectory + tracking MC states...")
            t0 = time.time()
            full_syn, states_arr = generate_with_states(needed_length, init_state, init_segment)
            np.save(syn_path, full_syn)
            np.save(states_path, states_arr)
            print(f"  Generation: {(time.time()-t0)/60:.1f} min  (states saved: {states_arr.shape})")

        def build_curr_dataset(ey):
            if ey == 0:
                return copy_dataset_downstream(init_dataset)
            syn_chunk = full_syn[:ey * AUG_LENGTH]
            syn_dataset = create_syn_dataset(syn_chunk, WINDOW_SIZE, scaler, DATATYPE)
            return concat_datasets_downstream(copy_dataset_downstream(init_dataset), syn_dataset)

    # ========================================================================
    # PROTOCOL: 'random_init'
    # ========================================================================
    elif protocol == 'random_init':
        blocks_path = f'{SYN_DIR}/run_{run_idx:02d}_blocks.npy'
        n_blocks_needed = max(eval_years)

        cached_blocks = None
        if os.path.exists(blocks_path):
            cached_blocks = np.load(blocks_path)
            n_existing = cached_blocks.shape[0] if cached_blocks.ndim == 2 else 0
            print(f"  Loading {n_existing} cached blocks (random init) for run {run_idx}")
        else:
            n_existing = 0

        if n_existing < n_blocks_needed:
            n_to_generate = n_blocks_needed - n_existing
            print(f"  Generating {n_to_generate} new 252-day blocks (random init each, ~{n_to_generate*0.05:.1f} min)...")
            t0 = time.time()
            new_blocks = []
            rng_local = np.random.RandomState(seed_base + run_idx + 9999)
            for b in range(n_existing, n_blocks_needed):
                # Sample ONLY from the LSTM-train portion (first 62.5% of FTS-Diffusion test set)
                # to avoid leaking the LSTM test period into the synthetic init.
                rand_idx = rng_local.randint(0, len(SEGMENTS_INIT))
                rand_init_segment = SEGMENTS_INIT[rand_idx]
                rand_init_pattern = LABELS_INIT[rand_idx]
                rand_init_length = LENGTHS_INIT[rand_idx]
                rand_init_magnitude = max(rand_init_segment) - min(rand_init_segment)
                rand_init_state = np.stack(
                    (rand_init_pattern, rand_init_length, rand_init_magnitude), axis=0)
                rand_init_state = torch.tensor(rand_init_state).view(1, -1)

                block = generate_timeseries_ftsdiffusion(
                    T=AUG_LENGTH,
                    init_state=rand_init_state,
                    init_segment=rand_init_segment)
                block = np.asarray(block, dtype=np.float32)[:AUG_LENGTH]
                if len(block) < AUG_LENGTH:
                    block = np.concatenate([block, np.full(AUG_LENGTH - len(block),
                                                          block[-1] if len(block) > 0 else 0.0)])
                new_blocks.append(block)
                if (b + 1) % 10 == 0:
                    stacked = np.stack(new_blocks, axis=0)
                    full = np.concatenate([cached_blocks, stacked], axis=0) if cached_blocks is not None else stacked
                    np.save(blocks_path, full)
            new_blocks_arr = np.stack(new_blocks, axis=0)
            all_blocks = np.concatenate([cached_blocks, new_blocks_arr], axis=0) if cached_blocks is not None else new_blocks_arr
            np.save(blocks_path, all_blocks)
            print(f"  Block generation: {(time.time()-t0)/60:.1f} min")
        else:
            all_blocks = cached_blocks

        def build_curr_dataset(ey):
            if ey == 0:
                return copy_dataset_downstream(init_dataset)
            curr = copy_dataset_downstream(init_dataset)
            for b in range(ey):
                block = all_blocks[b]
                block_dataset = create_syn_dataset(block, WINDOW_SIZE, scaler, DATATYPE)
                curr = concat_datasets_downstream(curr, block_dataset)
            return curr

    # ========================================================================
    # PROTOCOL: 'split' \u2014 1 long continuous trajectory + STATE TRACKING
    # ========================================================================
    elif protocol == 'split':
        cont_path = f'{SYN_DIR}/run_{run_idx:02d}_continuous.npy'
        states_path = f'{SYN_DIR}/run_{run_idx:02d}_states.npy'
        n_blocks_needed = max(eval_years)
        needed_length = n_blocks_needed * AUG_LENGTH

        full_syn = None
        if os.path.exists(cont_path) and os.path.exists(states_path):
            existing = np.load(cont_path)
            if len(existing) >= needed_length:
                print(f"  Loading saved continuous trajectory + states for run {run_idx} (protocol={protocol})")
                full_syn = existing

        if full_syn is None:
            print(f"  Generating one {n_blocks_needed}-year continuous trajectory + tracking MC states...")
            t0 = time.time()
            full_syn, states_arr = generate_with_states(needed_length, init_state, init_segment)
            np.save(cont_path, full_syn)
            np.save(states_path, states_arr)
            print(f"  Generation: {(time.time()-t0)/60:.1f} min  (states saved: {states_arr.shape})")

        all_blocks = np.array([
            full_syn[b*AUG_LENGTH:(b+1)*AUG_LENGTH]
            for b in range(n_blocks_needed)
        ], dtype=np.float32)

        def build_curr_dataset(ey):
            if ey == 0:
                return copy_dataset_downstream(init_dataset)
            curr = copy_dataset_downstream(init_dataset)
            for b in range(ey):
                block = all_blocks[b]
                block_dataset = create_syn_dataset(block, WINDOW_SIZE, scaler, DATATYPE)
                curr = concat_datasets_downstream(curr, block_dataset)
            return curr


    # ========================================================================
    # PROTOCOL: 'burn_in' \u2014 long continuous trajectory with burn-in discard,
    # then split into Y-year chunks (unfolded separately, like authors' protocol).
    # Pedagogical purpose: probe whether the FTS-Diffusion MC has a stationary
    # regime and how long it takes to reach it. With B=0,Y=1 this should be
    # IDENTICAL to the 'split' protocol (sanity check).
    # Total windows for k years of useful data: roughly (Y*252-63) * ceil(k/Y).
    # ========================================================================
    elif protocol == 'burn_in':
        cont_path   = f'{SYN_DIR}/run_{run_idx:02d}_continuous.npy'
        states_path = f'{SYN_DIR}/run_{run_idx:02d}_states.npy'

        T_target_years = max(eval_years)              # useful years requested
        total_years    = B_YEARS + T_target_years     # B burn-in + T useful
        needed_length  = total_years * AUG_LENGTH

        full_syn = None
        if os.path.exists(cont_path) and os.path.exists(states_path):
            existing = np.load(cont_path)
            if len(existing) >= needed_length:
                print(f"  Loading saved trajectory + states for run {run_idx} (protocol=burn_in, B={B_YEARS}y)")
                full_syn = existing

        if full_syn is None:
            print(f"  Generating {total_years}y ({B_YEARS}y burn-in + {T_target_years}y useful) + tracking MC states...")
            t0 = time.time()
            full_syn, states_arr = generate_with_states(needed_length, init_state, init_segment)
            np.save(cont_path, full_syn)
            np.save(states_path, states_arr)
            print(f"  Generation: {(time.time()-t0)/60:.1f} min  (states: {states_arr.shape})")

        # Discard the burn-in days
        syn_useful = full_syn[B_YEARS * AUG_LENGTH:]   # length >= T_target_years * 252

        # Build the dataset for evaluation year ey:
        #   take the first ey years of useful synthetic data
        #   split into Y-year chunks
        #   unfold each chunk separately and concatenate (no Frankenstein windows)
        # If ey % Y != 0, include a partial final chunk of (ey % Y) years.
        def build_curr_dataset(ey):
            if ey == 0:
                return copy_dataset_downstream(init_dataset)
            curr = copy_dataset_downstream(init_dataset)
            n_full_chunks  = ey // Y_YEARS
            partial_years  = ey % Y_YEARS
            chunk_days     = Y_YEARS * AUG_LENGTH
            # Full Y-year chunks
            for c in range(n_full_chunks):
                start = c * chunk_days
                chunk = syn_useful[start:start + chunk_days]
                if len(chunk) >= WINDOW_SIZE:
                    block_dataset = create_syn_dataset(chunk, WINDOW_SIZE, scaler, DATATYPE)
                    curr = concat_datasets_downstream(curr, block_dataset)
            # Partial chunk (last few years, if any)
            if partial_years > 0:
                start = n_full_chunks * chunk_days
                partial_days = partial_years * AUG_LENGTH
                chunk = syn_useful[start:start + partial_days]
                if len(chunk) >= WINDOW_SIZE:
                    block_dataset = create_syn_dataset(chunk, WINDOW_SIZE, scaler, DATATYPE)
                    curr = concat_datasets_downstream(curr, block_dataset)
            return curr

    else:
        raise ValueError(f"Unknown protocol: {protocol}. Use 'authors', 'single', 'random_init', 'split', or 'burn_in'.")

    # === Per-year LSTM training & evaluation ===
    for ey in eval_years:
        if ey in results:
            print(f"  Run {run_idx:2d} | Year {ey:3d} | already done \u2014 MAPE={results[ey]:.5f} (loaded)")
            continue

        curr_dataset = build_curr_dataset(ey)

        t0 = time.time()
        model_lstm, train_loss = train_lstm_full_batch(curr_dataset, seed=seed_base + run_idx + ey)
        train_time = time.time() - t0
        mape = evaluate_lstm(model_lstm, test_dataset, scaler)
        results[ey] = mape

        print(f"  Run {run_idx:2d} | Year {ey:3d} | windows={curr_dataset.shape[0]:6d} | "
              f"train_loss={train_loss:.4f} | MAPE={mape:.5f} | LSTM={train_time:.1f}s")

        sorted_years = sorted(results.keys())
        df_save = pd.DataFrame({
            'eval_year': sorted_years,
            'mape': [results[y] for y in sorted_years]
        })
        df_save.to_csv(run_csv, index=False)

        del model_lstm, curr_dataset
        torch.cuda.empty_cache()

    return {ey: results[ey] for ey in eval_years if ey in results}

print(f"\u2713 TATR runner ready (protocol={PROTOCOL!r}, MC state tracking enabled for 'single' & 'split')")


In [ ]:
# ============================================================================
# === MAIN LOOP \u2014 runs all replicates with live plotting + mean tracking ===
# ============================================================================
%matplotlib inline
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

all_results = {}
running_mape = np.full((N_RUNS, len(EVAL_YEARS)), np.nan)

# === STEP 1: Pre-load any runs already saved on Drive ===
print(f"Pre-loading existing runs from {RES_DIR}...")
for run_idx in range(N_RUNS):
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if os.path.exists(run_csv):
        df = pd.read_csv(run_csv)
        for j, ey in enumerate(EVAL_YEARS):
            mask = df['eval_year'] == ey
            if mask.any():
                running_mape[run_idx, j] = df.loc[mask, 'mape'].values[0]
        all_results[run_idx] = dict(zip(df['eval_year'], df['mape']))
n_preloaded = int(sum(~np.all(np.isnan(running_mape), axis=1)))
print(f"  \u2713 Found {n_preloaded} existing runs on Drive\n")


# === STEP 2: Mean tracker ===
mean_history = []

def update_mean_history():
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return None
    means = np.nanmean(running_mape[valid], axis=0)
    snap = {'n_runs': n_done}
    for j, ey in enumerate(EVAL_YEARS):
        snap[f'mean_y{ey}'] = float(means[j]) if not np.isnan(means[j]) else None
    mean_history.append(snap)
    return means


# === STEP 3: Live plot helper ===
def plot_live(elapsed_min=None):
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(12, 7))

    for r in range(N_RUNS):
        if not valid[r]:
            continue
        # Skip NaN points (years not yet computed for this run)
        mask = ~np.isnan(running_mape[r])
        ax.plot(np.array(EVAL_YEARS)[mask], running_mape[r][mask],
                color='steelblue', alpha=0.3, linewidth=1)

    means = np.nanmean(running_mape[valid], axis=0) if n_done >= 1 else np.full(len(EVAL_YEARS), np.nan)
    if n_done >= 2:
        stds = np.nanstd(running_mape[valid], axis=0)
        ax.fill_between(EVAL_YEARS, means - stds, means + stds,
                        alpha=0.25, color='steelblue', label=f'\u00b11 std (n={n_done})')
    valid_mean = ~np.isnan(means)
    ax.plot(np.array(EVAL_YEARS)[valid_mean], means[valid_mean],
            'o-', linewidth=2.5, color='darkblue', markersize=8, label='Mean MAPE')
    if not np.isnan(means[0]):
        ax.axhline(means[0], color='gray', linestyle='--', alpha=0.7,
                   label=f'Year-0 baseline = {means[0]:.4f}')

    for x, y in zip(EVAL_YEARS, means):
        if np.isnan(y):
            continue
        ax.annotate(f'{y:.4f}', xy=(x, y), xytext=(0, 12),
                    textcoords='offset points', ha='center',
                    fontsize=9, fontweight='bold', color='darkblue',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white',
                              ec='darkblue', alpha=0.85, linewidth=0.8))

    title = f'{EXPERIMENT.upper()} live \u2014 {PRETTY_NAME} (K={K}, protocol={PROTOCOL!r}) \u2014 {n_done}/{N_RUNS} runs done'
    if elapsed_min is not None:
        title += f' (last: {elapsed_min:.1f} min)'
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Augmentation (years)', fontsize=11)
    ax.set_ylabel('MAPE on real test set', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax + 0.015)
    plt.tight_layout()

    # Save current state to Drive (OVERWRITES — single live snapshot per asset+experiment)
    fig.savefig(f'{FIG_DIR}/live.png', dpi=120, bbox_inches='tight')

    display(fig)
    plt.close(fig)


# === STEP 4: Mean delta printer ===
def print_mean_delta(prev_means, new_means, n_done):
    if prev_means is None or new_means is None:
        if new_means is not None:
            print(f"\n[Mean tracking] First run completed (n=1). Initial means:")
            for ey, m in zip(EVAL_YEARS, new_means):
                if not np.isnan(m):
                    print(f"  Year {ey:3d}: {m:.5f}")
        return
    print(f"\n[Mean tracking] After run {n_done} (n={n_done}):")
    print(f"  {'Year':>4} | {'Old mean':>10} | {'New mean':>10} | {'\u0394':>10}")
    print(f"  {'-'*4}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}")
    for ey, old, new in zip(EVAL_YEARS, prev_means, new_means):
        if np.isnan(old) and np.isnan(new):
            print(f"  {ey:>4d} | {'NaN':>10} | {'NaN':>10} | {'\u2014':>10}")
            continue
        if np.isnan(old):
            print(f"  {ey:>4d} | {'NaN':>10} | {new:>10.5f} | NEW")
            continue
        if np.isnan(new):
            print(f"  {ey:>4d} | {old:>10.5f} | {'NaN':>10} | {'\u2014':>10}")
            continue
        delta = new - old
        sign = '\u2193' if delta < 0 else ('\u2191' if delta > 0 else '=')
        print(f"  {ey:>4d} | {old:>10.5f} | {new:>10.5f} | {sign} {abs(delta):.5f}")


# === STEP 5: Initialize history with pre-loaded data ===
prev_means = update_mean_history()
plot_live()


# === STEP 6: Main run loop ===
for run_idx in range(N_RUNS):
    print(f"\n{'='*70}")
    print(f"Run {run_idx + 1}/{N_RUNS} \u2014 {PRETTY_NAME} ({EXPERIMENT.upper()})")
    print(f"{'='*70}")

    t0 = time.time()
    results = run_single_tatr(run_idx, EVAL_YEARS)
    all_results[run_idx] = results
    elapsed = (time.time() - t0) / 60

    # === FIX: handle missing years gracefully (no KeyError) ===
    for j, ey in enumerate(EVAL_YEARS):
        running_mape[run_idx, j] = results.get(ey, np.nan)

    new_means = update_mean_history()
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    print_mean_delta(prev_means, new_means, n_done)
    prev_means = new_means.copy() if new_means is not None else None

    plot_live(elapsed_min=elapsed)
    pd.DataFrame(mean_history).to_csv(f'{RES_DIR}/mean_history.csv', index=False)

    print(f"\u2713 Run {run_idx + 1} done in {elapsed:.1f} min")

print(f"\n{'='*70}\nALL {N_RUNS} RUNS COMPLETE for {PRETTY_NAME} ({EXPERIMENT.upper()})\n{'='*70}")

## 5. Phase 3 — Aggregation & Confidence Intervals

In [ ]:
# Reload all per-run CSVs from Drive (handles re-runs, partial completion)
rows = []
for run_idx in range(N_RUNS):
    p = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if not os.path.exists(p):
        print(f"\u26a0 Missing run {run_idx}")
        continue
    df = pd.read_csv(p)
    df['run_idx'] = run_idx
    rows.append(df)

df_all = pd.concat(rows, ignore_index=True)
df_pivot = df_all.pivot(index='run_idx', columns='eval_year', values='mape')
print(f"Results matrix: {df_pivot.shape} (runs \u00d7 eval_years)")
print("\nMAPE per run \u00d7 evaluation year:")
print(df_pivot.round(5))

In [ ]:
# Compute statistics: mean, 95% CI via percentile bootstrap
from scipy import stats as sstats

def bootstrap_ci(x, n_boot=10000, ci=0.95, seed=0):
    """Percentile bootstrap CI for the mean."""
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [(1-ci)/2, 1-(1-ci)/2])
    return lo, hi

summary_stats = []
for ey in EVAL_YEARS:
    vals = df_pivot[ey].dropna().values
    mean = vals.mean()
    std = vals.std()
    lo, hi = bootstrap_ci(vals)
    summary_stats.append({
        'eval_year': ey,
        'mean_mape': mean,
        'std_mape': std,
        'ci95_low': lo,
        'ci95_high': hi,
        'n_runs': len(vals)
    })

summary_df = pd.DataFrame(summary_stats)
print("\nSummary statistics (95% bootstrap CI over runs):")
print(summary_df.round(5))

# Save
summary_df.to_csv(f'{RES_DIR}/summary.csv', index=False)
df_pivot.to_csv(f'{RES_DIR}/results_matrix.csv')
print(f"\n\u2713 Saved to {RES_DIR}/summary.csv and results_matrix.csv")

## 6. Final Figure — Replicate Fig. 6(b) of the Paper

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

x = summary_df['eval_year'].values * AUG_LENGTH
mean = summary_df['mean_mape'].values
lo = summary_df['ci95_low'].values
hi = summary_df['ci95_high'].values

ax.plot(x, mean, 'o-', linewidth=2.5, color='steelblue', markersize=7,
        label=f'FTS-Diffusion (n={N_RUNS} runs)')
ax.fill_between(x, lo, hi, alpha=0.25, color='steelblue', label='95% bootstrap CI')
ax.axhline(mean[0], color='gray', linestyle='--', alpha=0.7,
           label=f'Initial MAPE (year 0) = {mean[0]:.4f}')

# Annotate mean on each marker
for xv, yv in zip(x, mean):
    ax.annotate(f'{yv:.4f}', xy=(xv, yv), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=9,
                fontweight='bold', color='darkblue',
                bbox=dict(boxstyle='round,pad=0.3', fc='white',
                          ec='darkblue', alpha=0.85, linewidth=0.8))

if EVAL_YEARS[-1] == 100:
    pct_change = 100 * (mean[-1] - mean[0]) / mean[0]
    ax.annotate(f'\u0394MAPE @ year 100: {pct_change:+.1f}%\n(paper: -17.9%)',
                xy=(x[-1], mean[-1]), xytext=(x[-1]*0.55, mean[-1] + 0.02),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=11, color='red')

ax.set_xlabel('Augmented Size (days)', fontsize=12)
ax.set_ylabel('MAPE (real test set)', fontsize=12)
ax.set_title(f'{EXPERIMENT.upper()} \u2014 {PRETTY_NAME} (K={K}, {PROTOCOL!r}) \u2014 {N_RUNS} runs \u00d7 {len(EVAL_YEARS)} eval points '
             f'= {N_RUNS*len(EVAL_YEARS)} LSTMs', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 0.015)
plt.tight_layout()

fig_path = f'{FIG_DIR}/final.pdf'
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.savefig(fig_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
plt.show()
print(f"\u2713 Figure saved to {fig_path}")

## 7. Quality Checks

In [ ]:
# === Sanity checks ===
print("\n" + "="*70)
print("SANITY CHECKS")
print("="*70)

# 1) Every run completed?
print(f"\n[1] Runs completed: {len(df_pivot)}/{N_RUNS}")

# 2) Per-eval-year counts (should all be N_RUNS)
print(f"\n[2] Run counts per eval year:")
print(df_pivot.notna().sum(axis=0).to_string())

# 3) Trend: mean MAPE should generally DECREASE with more synthetic data (paper claim)
print(f"\n[3] Trend check — mean MAPE per eval year:")
print(summary_df[['eval_year', 'mean_mape']].to_string(index=False))
trend = np.polyfit(summary_df['eval_year'], summary_df['mean_mape'], 1)
print(f"  Linear trend slope: {trend[0]:.2e} ({'\u2193 downward (good)' if trend[0] < 0 else '\u2191 upward (NOT matching paper)'})")

# 4) Variance: across-run std should be reasonable (not all runs collapsing)
print(f"\n[4] Across-run std at each eval year:")
print(summary_df[['eval_year', 'std_mape']].to_string(index=False))

# 5) Year 0 baseline — paper reports ~0.026-0.05 MAPE
y0_mean = summary_df.iloc[0]['mean_mape']
print(f"\n[5] Year-0 baseline MAPE: {y0_mean:.5f} (paper Fig.6(b) starts ~0.05)")

# 6) Compute % reduction at year 100
if EVAL_YEARS[-1] == 100:
    y100_mean = summary_df.iloc[-1]['mean_mape']
    pct = 100 * (y0_mean - y100_mean) / y0_mean
    print(f"\n[6] MAPE reduction year 0 \u2192 100: {pct:+.2f}% (paper: 17.9%)")

## 8. Confronti finali: cross-protocol e cross-K

Queste celle caricano in automatico tutti i `summary.csv` disponibili su Drive e producono plot di confronto. Le celle scoprono autonomamente quali combinazioni di (ASSET, K, PROTOCOL) hai effettivamente eseguito, quindi puoi lanciarle in qualunque momento — anche dopo aver completato solo alcuni esperimenti.

### 8.1 Helper functions e discovery

In [ ]:
# === Helper functions for cross-comparison plots ===
import glob

def load_summary(asset, k, exp, protocol):
    """Load summary.csv for a (asset, K, exp, protocol) combination, or None."""
    path = f'{PERSIST_ROOT}/results/{exp}/{asset}/k{k}/{protocol}/summary.csv'
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    return df if len(df) > 0 else None


def discover_available(exp='tatr'):
    """Scan Drive for all (asset, K, protocol) triples that have a summary.csv."""
    base = f'{PERSIST_ROOT}/results/{exp}'
    if not os.path.exists(base):
        return {}
    found = {}  # {asset: {K: [protocols]}}
    for asset in sorted(os.listdir(base)):
        asset_path = os.path.join(base, asset)
        if not os.path.isdir(asset_path):
            continue
        found[asset] = {}
        for k_folder in sorted(os.listdir(asset_path)):
            if not (k_folder.startswith('k') and k_folder[1:].isdigit()):
                continue
            k = int(k_folder[1:])
            k_path = os.path.join(asset_path, k_folder)
            protocols = []
            for p in sorted(os.listdir(k_path)):
                if os.path.isdir(os.path.join(k_path, p)):
                    if os.path.exists(os.path.join(k_path, p, 'summary.csv')):
                        protocols.append(p)
            if protocols:
                found[asset][k] = protocols
    return found


# Show what's available
available = discover_available(EXPERIMENT)
print(f"Available {EXPERIMENT.upper()} results on Drive:\n")
if not available:
    print("  (none yet \u2014 run Phase 3 aggregation for at least one config first)")
else:
    for asset, ks in available.items():
        pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)
        print(f"  \u2022 {asset} ({pretty}):")
        for k, ps in ks.items():
            print(f"      K={k}: {ps}")

### 8.2 Cross-protocol comparison (same ASSET, same K, vary PROTOCOL)

In [ ]:
# === Cross-protocol comparison plot ===

PROTOCOL_COLORS = {
    'authors':     'steelblue',
    'single':      'coral',
    'random_init': 'forestgreen',
    'split':       'purple',
}

def plot_cross_protocol(asset, k, exp='tatr', annotate_values=True):
    """Overlay all available protocols for a fixed (asset, K)."""
    base = f'{PERSIST_ROOT}/results/{exp}/{asset}/k{k}'
    if not os.path.exists(base):
        print(f"\u26a0 No results for {asset} k={k}")
        return

    summaries = {}
    for p in os.listdir(base):
        s = load_summary(asset, k, exp, p)
        if s is not None:
            summaries[p] = s

    if not summaries:
        print(f"\u26a0 No summary.csv files for {asset} k={k}")
        return

    fig, ax = plt.subplots(figsize=(13, 7))
    pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)

    for p, s in sorted(summaries.items()):
        x = s['eval_year'].values * AUG_LENGTH
        mean = s['mean_mape'].values
        lo = s['ci95_low'].values
        hi = s['ci95_high'].values
        n = int(s['n_runs'].iloc[0])
        c = PROTOCOL_COLORS.get(p, 'gray')
        ax.plot(x, mean, 'o-', linewidth=2.2, color=c, markersize=7,
                label=f'{p} (n={n})')
        ax.fill_between(x, lo, hi, alpha=0.18, color=c)
        if annotate_values:
            for xv, yv in zip(x, mean):
                if np.isnan(yv): continue
                ax.annotate(f'{yv:.3f}', xy=(xv, yv), xytext=(0, 9),
                            textcoords='offset points', ha='center',
                            fontsize=7, color=c, alpha=0.9,
                            bbox=dict(boxstyle='round,pad=0.2', fc='white',
                                      ec=c, alpha=0.7, linewidth=0.4))

    ax.set_xlabel('Augmented Size (days)', fontsize=12)
    ax.set_ylabel('MAPE (real test set)', fontsize=12)
    ax.set_title(f'{exp.upper()} \u2014 {pretty} (K={k}) \u2014 protocol comparison',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    save_dir = f'{PERSIST_ROOT}/figures/comparisons/{asset}/k{k}'
    os.makedirs(save_dir, exist_ok=True)
    pdf_path = f'{save_dir}/cross_protocol.pdf'
    fig.savefig(pdf_path, bbox_inches='tight', dpi=150)
    fig.savefig(pdf_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\u2713 Saved {pdf_path}")


# === RUN: produce cross-protocol comparison for every (asset, K) found on Drive ===
for asset, ks in (available or {}).items():
    for k in ks:
        if len(ks[k]) >= 2:    # skip if only 1 protocol \u2014 nothing to compare
            print(f"\n--- {asset} K={k} ---")
            plot_cross_protocol(asset, k, exp=EXPERIMENT)
        else:
            print(f"\u26a0 Skipping {asset} K={k}: only 1 protocol available ({ks[k]})")

### 8.3 Cross-K comparison (same ASSET, same PROTOCOL, vary K)

In [ ]:
# === Cross-K comparison plot ===

def plot_cross_K(asset, protocol='authors', exp='tatr', annotate_values=True):
    """Overlay all available K values for a fixed (asset, protocol)."""
    base = f'{PERSIST_ROOT}/results/{exp}/{asset}'
    if not os.path.exists(base):
        print(f"\u26a0 No results for {asset}")
        return

    ks = []
    for k_folder in sorted(os.listdir(base)):
        if k_folder.startswith('k') and k_folder[1:].isdigit():
            k = int(k_folder[1:])
            if os.path.exists(f'{base}/{k_folder}/{protocol}/summary.csv'):
                ks.append(k)

    if len(ks) < 2:
        print(f"\u26a0 {asset} protocol={protocol}: only {len(ks)} K values available, skip")
        return

    fig, ax = plt.subplots(figsize=(13, 7))
    pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)

    cmap = plt.cm.viridis
    colors = [cmap(i / max(len(ks)-1, 1)) for i in range(len(ks))]

    for k, c in zip(ks, colors):
        s = load_summary(asset, k, exp, protocol)
        if s is None: continue
        x = s['eval_year'].values * AUG_LENGTH
        mean = s['mean_mape'].values
        lo = s['ci95_low'].values
        hi = s['ci95_high'].values
        n = int(s['n_runs'].iloc[0])
        ax.plot(x, mean, 'o-', linewidth=2.2, color=c, markersize=7,
                label=f'K={k} (n={n})')
        ax.fill_between(x, lo, hi, alpha=0.18, color=c)
        if annotate_values:
            for xv, yv in zip(x, mean):
                if np.isnan(yv): continue
                ax.annotate(f'{yv:.3f}', xy=(xv, yv), xytext=(0, 9),
                            textcoords='offset points', ha='center',
                            fontsize=7, color=c, alpha=0.9,
                            bbox=dict(boxstyle='round,pad=0.2', fc='white',
                                      ec=c, alpha=0.7, linewidth=0.4))

    ax.set_xlabel('Augmented Size (days)', fontsize=12)
    ax.set_ylabel('MAPE (real test set)', fontsize=12)
    ax.set_title(f'{exp.upper()} \u2014 {pretty} (protocol={protocol!r}) \u2014 K comparison',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    save_dir = f'{PERSIST_ROOT}/figures/comparisons/{asset}'
    os.makedirs(save_dir, exist_ok=True)
    pdf_path = f'{save_dir}/cross_K_{protocol}.pdf'
    fig.savefig(pdf_path, bbox_inches='tight', dpi=150)
    fig.savefig(pdf_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\u2713 Saved {pdf_path}")


# === RUN: produce cross-K comparison for every (asset, protocol) found on Drive ===
for asset, ks in (available or {}).items():
    # Find which protocols have multiple K values
    proto_to_ks = {}
    for k, prots in ks.items():
        for p in prots:
            proto_to_ks.setdefault(p, []).append(k)
    for p, kvals in proto_to_ks.items():
        if len(kvals) >= 2:
            print(f"\n--- {asset} protocol={p!r} (K={kvals}) ---")
            plot_cross_K(asset, protocol=p, exp=EXPERIMENT)
        else:
            print(f"\u26a0 Skipping {asset} protocol={p!r}: only K={kvals[0]} available")

### 8.4 Summary table

In [ ]:
# === Cross-everything summary table ===
# Numerical comparison across (asset, K, protocol)

rows = []
for asset, ks in (available or {}).items():
    pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)
    for k, prots in ks.items():
        for p in prots:
            s = load_summary(asset, k, EXPERIMENT, p)
            if s is None: continue
            y0 = s.iloc[0]
            y_max = s.iloc[-1]
            mean_y0 = float(y0['mean_mape'])
            mean_ymax = float(y_max['mean_mape'])
            pct_change = 100 * (mean_ymax - mean_y0) / mean_y0 if mean_y0 != 0 else float('nan')
            rows.append({
                'asset':     pretty,
                'K':         k,
                'protocol':  p,
                'n_runs':    int(y0['n_runs']),
                'mape_y0':   mean_y0,
                'mape_y100': mean_ymax,
                'pct_change': pct_change,
                'last_year': int(y_max['eval_year']),
            })

if rows:
    summary_all = pd.DataFrame(rows)
    summary_all = summary_all.sort_values(['asset', 'protocol', 'K']).reset_index(drop=True)
    print("\n=== Summary across (asset, K, protocol) ===\n")
    print(summary_all.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

    save_path = f'{PERSIST_ROOT}/figures/comparisons/cross_everything_summary.csv'
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    summary_all.to_csv(save_path, index=False)
    print(f"\n\u2713 Saved table to {save_path}")
else:
    print("No data yet for the summary table.")

## 9. Markov chain parameter evolution analysis

For the protocols `'single'` and `'split'`, we save the full state sequence $(p_t, l_t, m_t)$ along with the synthetic trajectory. This lets us analyze long-term stability of the FTS-Diffusion Markov chain (drift, attractors, stationary distribution, transitions). For `'authors'` and `'random_init'` the chain is reset every block, so this analysis is not meaningful and the cells below skip them automatically.

### 9.1 Helpers

In [ ]:
# === Helpers to load and analyze MC state trajectories ===

def load_states(asset, k, exp, protocol, run_idx):
    """Load (n_segments, 3) states array for a single run, or None."""
    base = f'{PERSIST_ROOT}/synthetic/{asset}/k{k}/{protocol}'
    path = f'{base}/run_{run_idx:02d}_states.npy'
    if os.path.exists(path):
        return np.load(path)
    return None


def load_all_states(asset, k, exp, protocol):
    """Return list of (run_idx, states_arr) for all runs that have states saved."""
    base = f'{PERSIST_ROOT}/synthetic/{asset}/k{k}/{protocol}'
    if not os.path.exists(base):
        return []
    out = []
    for f in sorted(os.listdir(base)):
        if f.startswith('run_') and f.endswith('_states.npy'):
            run_idx = int(f.split('_')[1])
            path = os.path.join(base, f)
            try:
                arr = np.load(path)
                if arr.ndim == 2 and arr.shape[1] == 3:
                    out.append((run_idx, arr))
            except Exception as e:
                print(f"  Skipped {path}: {e}")
    return out


def discover_state_data(exp='tatr'):
    """Find all (asset, K, protocol) triples that have at least one *_states.npy file."""
    base = f'{PERSIST_ROOT}/synthetic'
    if not os.path.exists(base):
        return []
    found = []
    for asset in sorted(os.listdir(base)):
        asset_path = os.path.join(base, asset)
        if not os.path.isdir(asset_path):
            continue
        for k_folder in sorted(os.listdir(asset_path)):
            if not (k_folder.startswith('k') and k_folder[1:].isdigit()):
                continue
            k = int(k_folder[1:])
            for p in ['single', 'split']:  # only protocols where states make sense
                proto_path = os.path.join(asset_path, k_folder, p)
                if not os.path.isdir(proto_path):
                    continue
                state_files = [f for f in os.listdir(proto_path) if f.endswith('_states.npy')]
                if state_files:
                    found.append((asset, k, p, len(state_files)))
    return found


# Show what's available
state_data = discover_state_data(EXPERIMENT)
print(f"Available MC state trajectories on Drive:\n")
if not state_data:
    print("  (none yet \u2014 run 'single' or 'split' protocol first)")
else:
    for asset, k, p, n_files in state_data:
        pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)
        print(f"  \u2022 {asset} ({pretty}) | K={k} | protocol={p!r}: {n_files} runs with states")

### 9.2 Plot MC state evolution per run (pattern, length, magnitude)

In [ ]:
# === Plot how (p, l, m) evolve over the segments of one trajectory ===

def plot_mc_evolution(asset, k, protocol, run_idx=0, save=True):
    """Plot pattern + length + magnitude vs segment index for one run."""
    states = load_states(asset, k, EXPERIMENT, protocol, run_idx)
    if states is None:
        print(f"\u26a0 No states for {asset} k{k} {protocol} run {run_idx}")
        return

    pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)
    n_seg = states.shape[0]
    seg_idx = np.arange(n_seg)
    p_seq = states[:, 0].astype(int)
    l_seq = states[:, 1]
    m_seq = states[:, 2]

    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

    # Pattern (categorical scatter)
    cmap = plt.cm.get_cmap('tab20', k)
    axes[0].scatter(seg_idx, p_seq, c=p_seq, cmap=cmap, s=8, alpha=0.7)
    axes[0].set_ylabel('Pattern $p_t$', fontsize=11)
    axes[0].set_title(f'MC state evolution \u2014 {pretty} K={k} {protocol!r} run {run_idx}',
                      fontsize=12, fontweight='bold')
    axes[0].set_yticks(range(k))
    axes[0].grid(alpha=0.3, axis='y')

    # Length
    axes[1].plot(seg_idx, l_seq, linewidth=0.8, color='steelblue', alpha=0.8)
    axes[1].set_ylabel('Length $l_t$ (days)', fontsize=11)
    axes[1].grid(alpha=0.3)

    # Magnitude
    axes[2].plot(seg_idx, m_seq, linewidth=0.8, color='coral', alpha=0.8)
    axes[2].set_xlabel('Segment index', fontsize=11)
    axes[2].set_ylabel('Magnitude $m_t$', fontsize=11)
    axes[2].grid(alpha=0.3)

    plt.tight_layout()

    if save:
        save_dir = f'{PERSIST_ROOT}/figures/mc_analysis/{asset}/k{k}/{protocol}'
        os.makedirs(save_dir, exist_ok=True)
        path = f'{save_dir}/state_evolution_run_{run_idx:02d}.pdf'
        fig.savefig(path, bbox_inches='tight', dpi=150)
        fig.savefig(path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
        print(f"  \u2713 Saved {path}")
    plt.show()
    plt.close(fig)


# Run for the FIRST run of every available (asset, K, protocol) combo
for asset, k, p, n_files in state_data:
    plot_mc_evolution(asset, k, p, run_idx=0, save=True)

### 9.3 Pattern transition matrix and stationary distribution

In [ ]:
# === Compute empirical transition matrix and stationary distribution ===

def compute_transition_matrix(p_sequence, K):
    """Empirical P(p_{t+1} | p_t) from a 1D sequence of pattern indices."""
    counts = np.zeros((K, K), dtype=np.float64)
    for t in range(len(p_sequence) - 1):
        counts[p_sequence[t], p_sequence[t+1]] += 1
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    return counts / row_sums


def plot_mc_distributions(asset, k, protocol, save=True):
    """For a given (asset, K, protocol):
       - Aggregate states across ALL runs available
       - Plot transition matrix (averaged over runs)
       - Plot stationary distribution of patterns
    """
    runs_states = load_all_states(asset, k, EXPERIMENT, protocol)
    if not runs_states:
        print(f"\u26a0 No states for {asset} k{k} {protocol}")
        return

    pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)
    n_runs = len(runs_states)

    # === Aggregate over all runs ===
    all_p = np.concatenate([s[:, 0].astype(int) for _, s in runs_states])
    all_l = np.concatenate([s[:, 1] for _, s in runs_states])
    all_m = np.concatenate([s[:, 2] for _, s in runs_states])

    # Transition matrix: average across runs
    transitions = np.zeros((k, k))
    for _, s in runs_states:
        transitions += compute_transition_matrix(s[:, 0].astype(int), k)
    transitions /= n_runs

    # Stationary distribution from aggregated visits
    stationary = np.bincount(all_p, minlength=k) / len(all_p)

    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 3, width_ratios=[2, 1.4, 1.4])

    # Transition matrix heatmap
    ax0 = fig.add_subplot(gs[0, 0])
    im = ax0.imshow(transitions, cmap='Blues', vmin=0, vmax=transitions.max(),
                    aspect='auto')
    ax0.set_xlabel('Next pattern $p_{t+1}$')
    ax0.set_ylabel('Current pattern $p_t$')
    ax0.set_title(f'Transition matrix (avg over {n_runs} runs)')
    ax0.set_xticks(range(k)); ax0.set_yticks(range(k))
    plt.colorbar(im, ax=ax0)
    # annotate
    for i in range(k):
        for j in range(k):
            v = transitions[i, j]
            if v > 0.05:
                ax0.text(j, i, f'{v:.2f}', ha='center', va='center',
                         fontsize=7, color='white' if v > 0.5 else 'black')

    # Stationary distribution
    ax1 = fig.add_subplot(gs[0, 1])
    cmap = plt.cm.get_cmap('tab20', k)
    ax1.bar(range(k), stationary, color=[cmap(i) for i in range(k)])
    ax1.set_xlabel('Pattern')
    ax1.set_ylabel('Frequency')
    ax1.set_title(f'Stationary distribution\n(aggregated over {n_runs} runs)')
    ax1.set_xticks(range(k))
    ax1.grid(alpha=0.3, axis='y')

    # Length / Magnitude histograms
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(all_l, bins=range(int(all_l.min()), int(all_l.max())+2),
             alpha=0.6, color='steelblue', label='Length $l_t$')
    ax2.set_xlabel('Length (days)')
    ax2.set_ylabel('Count', color='steelblue')
    ax2.tick_params(axis='y', labelcolor='steelblue')
    ax2_twin = ax2.twinx()
    ax2_twin.hist(all_m, bins=40, alpha=0.4, color='coral', label='Magnitude $m_t$')
    ax2_twin.set_ylabel('Count (magnitude)', color='coral')
    ax2_twin.tick_params(axis='y', labelcolor='coral')
    ax2.set_title('Length & magnitude distributions')
    ax2.grid(alpha=0.3)

    fig.suptitle(f'MC analysis \u2014 {pretty} K={k} protocol={protocol!r}',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()

    if save:
        save_dir = f'{PERSIST_ROOT}/figures/mc_analysis/{asset}/k{k}/{protocol}'
        os.makedirs(save_dir, exist_ok=True)
        path = f'{save_dir}/transition_and_stationary.pdf'
        fig.savefig(path, bbox_inches='tight', dpi=150)
        fig.savefig(path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
        print(f"  \u2713 Saved {path}")
    plt.show()
    plt.close(fig)

    return transitions, stationary


# Run for every (asset, K, protocol) with state data
for asset, k, p, n_files in state_data:
    print(f"\n--- {asset} K={k} protocol={p!r} ({n_files} runs) ---")
    plot_mc_distributions(asset, k, p, save=True)

### 9.4 Long-term drift: stability check across the trajectory

In [ ]:
# === Drift analysis: is the MC stationary? Compare distributions early vs late in the trajectory ===

def plot_mc_drift(asset, k, protocol, run_idx=0, save=True, n_chunks=4):
    """Split a single trajectory into n_chunks consecutive segments and compare
    pattern frequencies. If stationary, all chunks should look similar."""
    states = load_states(asset, k, EXPERIMENT, protocol, run_idx)
    if states is None:
        print(f"\u26a0 No states for {asset} k{k} {protocol} run {run_idx}")
        return

    pretty = ASSETS_CONFIG.get(asset, {}).get('pretty', asset)
    p_seq = states[:, 0].astype(int)
    n = len(p_seq)
    chunk_size = n // n_chunks

    fig, ax = plt.subplots(figsize=(13, 5))
    cmap = plt.cm.viridis
    colors = [cmap(i / max(n_chunks-1, 1)) for i in range(n_chunks)]

    width = 0.8 / n_chunks
    x = np.arange(k)
    for ci in range(n_chunks):
        chunk = p_seq[ci*chunk_size:(ci+1)*chunk_size]
        freq = np.bincount(chunk, minlength=k) / max(len(chunk), 1)
        ax.bar(x + ci*width, freq, width=width,
               color=colors[ci], label=f'segs {ci*chunk_size}-{(ci+1)*chunk_size}')

    ax.set_xlabel('Pattern')
    ax.set_ylabel('Frequency')
    ax.set_xticks(x + width * (n_chunks-1) / 2)
    ax.set_xticklabels(range(k))
    ax.set_title(f'MC drift check \u2014 {pretty} K={k} {protocol!r} run {run_idx} \u2014 {n_chunks} chunks',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()

    if save:
        save_dir = f'{PERSIST_ROOT}/figures/mc_analysis/{asset}/k{k}/{protocol}'
        os.makedirs(save_dir, exist_ok=True)
        path = f'{save_dir}/drift_check_run_{run_idx:02d}.pdf'
        fig.savefig(path, bbox_inches='tight', dpi=150)
        fig.savefig(path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
        print(f"  \u2713 Saved {path}")
    plt.show()
    plt.close(fig)


# Run for the FIRST run of every available (asset, K, protocol)
for asset, k, p, n_files in state_data:
    plot_mc_drift(asset, k, p, run_idx=0, save=True, n_chunks=4)

## 12. Year-0 MAPE diagnostic: what causes the discrepancy with the paper?

The paper (Appendix E.4) reports a naive baseline MAPE of $\approx 0.046$ for S&P 500. Our year-0 baseline (LSTM trained only on real init data, no synthetic) varies by configuration. This diagnostic grid systematically toggles each plausible cause to find which combination reproduces $\approx 0.046$.

In [ ]:
# === TASK 4: Year-0 MAPE diagnostic grid ===
# Run a small grid over (scaler, datatype, init_period) to isolate which knob
# reproduces the paper\'s 0.046 baseline. This trains an LSTM 1 time per cell
# at year 0 only — no synthetic data — and reports MAPE.

from sklearn.preprocessing import MinMaxScaler, StandardScaler

def _build_dataset_custom(timeseries, window_size, scaler_obj):
    """Apply scaler (already fit) and unfold into rolling windows."""
    if scaler_obj is None:
        ts = torch.tensor(timeseries, dtype=torch.float32)
    else:
        ts_scaled = scaler_obj.transform(np.asarray(timeseries).reshape(-1, 1))
        ts = torch.tensor(ts_scaled, dtype=torch.float32).squeeze(1)
    return ts.unfold(0, window_size, 1).float()


def _train_and_eval_lstm_diagnostic(init_ts, test_ts, scaler_obj, datatype_label,
                                     window_size=64, hidden_dim=32, n_layers=2,
                                     n_epochs=100, lr=1e-2, criterion_str='mae'):
    """Train LSTM on init_ts (only real init, year 0), eval MAPE on test_ts."""
    # Optional return-transform
    if datatype_label == 'returns':
        init_ts = np.diff(init_ts) / init_ts[:-1]
        test_ts = np.diff(test_ts) / test_ts[:-1]
    elif datatype_label == 'log_returns':
        init_ts = np.diff(np.log(init_ts))
        test_ts = np.diff(np.log(test_ts))
    # else prices: use as-is

    # Fit scaler on init only (no test leakage)
    if scaler_obj is not None:
        scaler_obj.fit(np.asarray(init_ts).reshape(-1, 1))

    init_dataset_d = _build_dataset_custom(init_ts, window_size, scaler_obj)
    test_dataset_d = _build_dataset_custom(test_ts, window_size, scaler_obj)

    torch.manual_seed(42)
    model = LSTMPredictor(input_dim=1, hidden_dim=hidden_dim, output_dim=1,
                          n_layers=n_layers, device=device).to(device)
    criterion = set_loss_fn(criterion_str)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    X = init_dataset_d[:, :-1].unsqueeze(-1).to(device)
    y = init_dataset_d[:, -1:].to(device)
    for _ in range(n_epochs):
        model.train()
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        Xt = test_dataset_d[:, :-1].unsqueeze(-1).to(device)
        yt = test_dataset_d[:, -1:].numpy()
        yp = model(Xt).cpu().numpy()
    if scaler_obj is not None:
        yt = scaler_obj.inverse_transform(yt)
        yp = scaler_obj.inverse_transform(yp)
    return float(MAPE(yt, yp))


# === Build grid ===
# Need raw data again (before any unfold)
fts_full = load_actual_fts(ASSET).squeeze()
# Use authors\' EXACT setup: 80/20 split, then first 252*5 days as init, rest as test
n = len(fts_full)
split_idx = int(n * 0.8)
test_set_full = fts_full[split_idx:]            # downstream_timeseries equivalent
fixed_init_period = 252 * 5                      # paper\'s hardcoded init
init_ts_paper = test_set_full[:fixed_init_period]
test_ts_paper = test_set_full[fixed_init_period:]

# Grid axes
scalers_to_try = {
    'MinMax(-1,1)': MinMaxScaler(feature_range=(-1, 1)),
    'MinMax(0,1)':  MinMaxScaler(feature_range=(0, 1)),
    'Standard':     StandardScaler(),
    'None':         None,
}
datatypes_to_try = ['prices', 'returns', 'log_returns']

print("Running 4 scalers x 3 datatypes = 12 diagnostic configs (year 0 only)...")
print("Each LSTM training takes ~20-40 sec on A100. Total ~5-10 min.\n")

rows = []
for sc_name, sc in scalers_to_try.items():
    for dt in datatypes_to_try:
        try:
            sc_copy = type(sc)(**sc.get_params()) if sc is not None else None
            t0 = time.time()
            mape = _train_and_eval_lstm_diagnostic(init_ts_paper, test_ts_paper,
                                                    sc_copy, dt)
            elapsed = time.time() - t0
            print(f"  scaler={sc_name:14s} datatype={dt:11s} -> MAPE={mape:.5f}  ({elapsed:.1f}s)")
            rows.append({'scaler': sc_name, 'datatype': dt, 'mape': mape, 'sec': elapsed})
        except Exception as e:
            print(f"  scaler={sc_name:14s} datatype={dt:11s} -> ERROR: {e}")
            rows.append({'scaler': sc_name, 'datatype': dt, 'mape': float(\'nan\'), 'sec': 0})

diag_df = pd.DataFrame(rows)
diag_df['delta_to_paper'] = (diag_df['mape'] - 0.046).abs()
diag_df = diag_df.sort_values('delta_to_paper')

print("\n=== Closest configurations to paper\'s 0.046 ===")
print(diag_df.to_string(index=False))

# Save
save_dir = f'{PERSIST_ROOT}/figures/diagnostics'
os.makedirs(save_dir, exist_ok=True)
diag_df.to_csv(f'{save_dir}/year0_mape_grid.csv', index=False)
print(f"\n\u2713 Saved to {save_dir}/year0_mape_grid.csv")

print("\nINTERPRETATION:")
print("  - If top row is 'prices' + 'MinMax(-1,1)', our default setup is correct.")
print("  - If 'returns' is closer to paper, the paper might predict returns not prices.")
print("  - If no config gets close, the discrepancy is elsewhere (e.g. different test period).")


## 8. How to Resume / Switch Asset / Add TMTR

### Resume after Colab disconnect
1. Reopen this notebook
2. Re-run cells 1-3 (setup, imports, config)
3. Re-run Phase 1 — it detects existing checkpoints for the current ASSET on Drive and skips
4. Re-run the main TATR loop — it detects completed runs in `RES_DIR` and skips them

### Switch to a different asset
1. Change `ASSET = 'goog'` (or `'zcf'`) in the **Configuration** cell
2. Re-run from Phase 1 onwards
3. Phase 1 will train the architecture for the new asset (~1h on A100)
4. Phase 2 will run the experiment for that asset
5. Results stored separately under `{PERSIST_ROOT}/results/{exp}/{ASSET}/`

### Switch from TATR to TMTR
1. Change `EXPERIMENT = 'tmtr'` in Configuration
2. (You will need a TMTR-specific main loop \u2014 currently this notebook implements TATR. The folder structure is ready for TMTR results to be stored at `{PERSIST_ROOT}/results/tmtr/{ASSET}/`.)

### Drive storage layout
```
/content/drive/MyDrive/fts_diffusion/
  \u251c\u2500\u2500 architectures/{asset}/   \u2190 SISC + PGM + PEM checkpoints (per asset)
  \u251c\u2500\u2500 synthetic/{asset}/        \u2190 Per-run synthetic series .npy
  \u251c\u2500\u2500 results/{exp}/{asset}/    \u2190 Per-run MAPE CSVs + summary + mean_history
  \u2514\u2500\u2500 figures/{exp}/            \u2190 Final plots
```

## Caveats / Sources of Variability

Even with `torch.manual_seed`, exact reproducibility on different GPUs is hard:
- **Different GPU = different cuDNN kernels** \u2192 small numerical differences
- **Sampling pipeline uses randomness in the diffusion noise** \u2192 different seed = different synthetic series
- **Across-run variance** is the *intended* signal: it captures uncertainty from generative randomness

If your year-100 MAPE reduction is in the range -10% to -25%, you've replicated the paper qualitatively.